In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "test_notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf
from imrw import imread

from src.ml.build import (
    build_birefnet,
    build_lora_birefnet_for_inference,
    build_lora_birefnet_for_training,
)
from src.ml.inference.predict import auto_predict, predict

In [ ]:
cfg = OmegaConf.merge(
    OmegaConf.load("src/config/tune.yaml"),
    OmegaConf.load("src/config/model.yaml"),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

LORA_CKPT = None
IMAGE_PATH = "local_data/image/backpack_02.jpg"

In [ ]:
base_model = build_birefnet(cfg=cfg).to(device)
if LORA_CKPT is None:
    model = build_lora_birefnet_for_training(cfg=cfg, model=base_model)
else:
    model = build_lora_birefnet_for_inference(cfg=cfg, model=base_model, ckpt_path=LORA_CKPT)
model.eval()
print(
    f"total={model.stats['total']:,}  trainable={model.stats['trainable']:,}  "
    f"ratio={model.stats['trainable'] / model.stats['total']:.2%}"
)

In [ ]:
image = imread(IMAGE_PATH)
print(f"image: {image.shape}  dtype: {image.dtype}")

mask = predict(model=model, image=image, threshold=None)
print(f"mask: {mask.shape}  dtype: {mask.dtype}  min={mask.min()}  max={mask.max()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("predict")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
auto_mask = auto_predict(model=model, image=image, threshold=None)
print(f"auto_mask: {auto_mask.shape}  dtype: {auto_mask.dtype}  min={auto_mask.min()}  max={auto_mask.max()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("predict")
axes[2].imshow(auto_mask, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("auto_predict")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()